In [13]:
import pandas as pd

# process io_data_df
io_data_df = (r"C:\Users\20245179\OneDrive - TU Eindhoven\LLM_EngD_project\Intent recognition Comments\annotate_data_subset\comments_sustainability_4labeling_lbld_io.csv")
rows = []
with open(io_data_df, encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(',')
        text = ','.join(parts[:-4])  # Join all parts except last 4 as 'text'
        nu_words, lbl1, lbl2 = parts[-4:-1]
        rows.append({'text': text, 'nu_words': nu_words, 'lbl1': lbl1, 'lbl2': lbl2})

import pandas as pd
io_data_df = pd.DataFrame(rows)

io_data_df = io_data_df.iloc[1:].reset_index(drop=True)
import pandas as pd

# New column: value after '_', or empty string if no '_'
io_data_df['sentiment1'] = io_data_df['lbl1'].apply(lambda x: x.split('_')[1] if '_' in x else '')

# Update lbl1: keep value before '_' if present, else keep original
io_data_df['lbl1'] = io_data_df['lbl1'].apply(lambda x: x.split('_')[0] if '_' in x else x)

# all the rows that hhave "Criticism" in lbl1 mark senbtiment1 as "-1"
io_data_df.loc[io_data_df['lbl1'] == 'criticism', 'sentiment1'] = '-1'
# all the rows that have "Appreciation" in lbl1 mark sentiment1 as "1"
io_data_df.loc[io_data_df['lbl1'] == 'appreciation', 'sentiment1'] = '1'
# all the rows that have "inquiry" in lbl1 mark sentiment1 as "0"
io_data_df.loc[io_data_df['lbl1'] == 'inquiry', 'sentiment1'] = '0'
# all the rows that have "forward" in lbl1 mark sentiment1 as "0"
io_data_df.loc[io_data_df['lbl1'] == 'forward', 'sentiment1'] = '0'

#reepalce "n" in sentiment 1 as "0"
io_data_df['sentiment1'] = io_data_df['sentiment1'].replace('n', '0')
#replce "p" in sentiment1 as "1"
io_data_df['sentiment1'] = io_data_df['sentiment1'].replace('p', '1')






In [15]:
io_data_df["sentiment1"].value_counts()

sentiment1
0     110
1      30
-1     18
Name: count, dtype: int64

In [6]:
#### process calrlijn_data
import pandas as pd
carlijn_data = pd.read_excel(r"C:\Users\20245179\OneDrive - TU Eindhoven\Research Paper\sentiment pred\test_sentiment_data_262_comments_labeled.xlsx")
carlijn_data["PI_Sentiment"] = carlijn_data["PI_Sentiment"].str.lower()
carlijn_data["PI_Sentiment"] = carlijn_data["PI_Sentiment"].replace("neutral", "0")
carlijn_data["PI_Sentiment"] = carlijn_data["PI_Sentiment"].replace("positive", "1")
carlijn_data["PI_Sentiment"] = carlijn_data["PI_Sentiment"].replace("negative", "-1")

In [7]:
carlijn_data

,Comments,PI,PI_Sentiment,SI,SI_Sentiment,Unnamed: 5,Unnamed: 6,custom_index,PI Sentiment,Secondary Intent(SI),SI Sentiment
0,Je huur moet wel boven de 575 zijn ze verlagen...,Statement,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Nemen jullie de telefoon nog op?,Inquiry,0,Criticism,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Hopelijk zo'n project weer in Stratum, dan kun...",Inquiry,1,NaN,NaN,can be positive and negative (little sarcasm),NaN,NaN,NaN,NaN,NaN
3,Echt slecht je loopt alleen maar tegen problem...,Criticism,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Zo zijn ze ook bij mijn moeder geweest... alth...,Criticism,-1,Statement,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
257,Dan kom ik langs voor sleutels en krijg er gra...,Inquiry,0,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN
258,Tussendeuren in huis gesloten houden ??? moet ...,Statement,-1,NaN,NaN,NaN,NaN,144.0,NaN,Inquiry,Negative
259,"Even een tip,volgens Dammers kloppen jullie pl...",Statement,0,NaN,NaN,NaN,NaN,97.0,NaN,NaN,NaN
260,Woonenergie regelt toch de sw woningbouwvereni...,Inquiry,0,NaN,NaN,NaN,NaN,140.0,NaN,NaN,NaN


In [8]:
carlijn_data_sent = carlijn_data[["Comments", "PI_Sentiment"]]

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from transformers import pipeline
from sklearn.metrics import accuracy_score
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
# Load dataset with 'text.y' and 'reewalk_label' columns
df = carlijn_data_sent
df = df[df["Comments"].notnull()].reset_index(drop=True)
df["PI_Sentiment"] = df["PI_Sentiment"].astype(int)

# Sentiment text to numeric label mapping
def to_numeric_label(label: str):
    label = label.lower()
    if label in ["positive", "pos", "label_1", "4 stars", "5 stars", "1", "very positive"]:
        return 1
    elif label in ["negative", "neg", "label_0", "1 star", "2 stars", "-1", "very negative"]:
        return -1
    else:
        return 0


# List of models to evaluate
models = [
    'DTAI-KULeuven/robbert-v2-dutch-sentiment',
    'pdelobelle/robbert-v2-dutch-base',
    'DTAI-KULeuven/robbertje-merged-dutch-sentiment',
    'nlptown/bert-base-multilingual-uncased-sentiment',
    'GroNLP/bert-base-dutch-cased',
    'citizenlab/twitter-xlm-roberta-base-sentiment-finetunned',
    "BramVanroy/xlm-roberta-base-hebban-reviews",
    "BramVanroy/bert-base-multilingual-cased-hebban-reviews",
    "BramVanroy/robbert-v2-dutch-base-hebban-reviews",
    "clips/republic",
]

results = {}

# Loop through models
for model_name in models:
    try:
        print(f"\n🔍 Loading model: {model_name}")
        classifier = pipeline("sentiment-analysis", model=model_name, return_all_scores=True, truncation=True, max_length=512)

        model_preds = []
        for text in df["Comments"]:
            try:
                scores = classifier(text)[0]
                if len(scores) == 2:
                    pos_score = scores[1]['score']
                    neg_score = scores[0]['score']
                    if abs(pos_score - neg_score) <= 0.1:
                        sentiment = "neutral"
                    elif pos_score > neg_score:
                        sentiment = "positive"
                    else:
                        sentiment = "negative"
                    model_preds.append(to_numeric_label(sentiment))

                else:
                    top = max(scores, key=lambda x: x["score"])
                    model_preds.append(to_numeric_label(top["label"]))
            except Exception as e:
                model_preds.append(0)

        df[model_name] = model_preds
        acc = accuracy_score(df["PI_Sentiment"], model_preds)
        print(f"✅ Accuracy: {acc:.4f}")
        print(classification_report(df["PI_Sentiment"], model_preds, digits=3))
        results[model_name] = acc
    

    except Exception as e:
        print(f"❌ Failed on {model_name}: {e}")

# Accuracy Summary
print("\n📊 Accuracy Summary:")
for model_name, acc in results.items():
    print(f"{model_name:65s} => {acc:.4f}")



🔍 Loading model: DTAI-KULeuven/robbert-v2-dutch-sentiment


Device set to use cpu
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at pdelobelle/robbert-v2-dutch-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Accuracy: 0.4198
              precision    recall  f1-score   support

          -1      0.678     0.713     0.695       115
           0      0.333     0.009     0.017       114
           1      0.196     0.818     0.316        33

    accuracy                          0.420       262
   macro avg      0.402     0.513     0.343       262
weighted avg      0.467     0.420     0.352       262


🔍 Loading model: pdelobelle/robbert-v2-dutch-base


Device set to use cpu


✅ Accuracy: 0.3397
              precision    recall  f1-score   support

          -1      0.387     0.583     0.465       115
           0      0.250     0.193     0.218       114
           1      0.000     0.000     0.000        33

    accuracy                          0.340       262
   macro avg      0.212     0.259     0.228       262
weighted avg      0.279     0.340     0.299       262


🔍 Loading model: DTAI-KULeuven/robbertje-merged-dutch-sentiment


...

✅ Accuracy: 0.6985
              precision    recall  f1-score   support

          -1      0.714     0.826     0.766       115
           0      0.843     0.518     0.641       114
           1      0.492     0.879     0.630        33

    accuracy                          0.698       262
   macro avg      0.683     0.741     0.679       262
weighted avg      0.742     0.698     0.695       262


🔍 Loading model: tabularisai/multilingual-sentiment-analysis


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:  19%|#9        | 105M/541M [00:00<?, ?B/s]

Device set to use cpu


✅ Accuracy: 0.7099
              precision    recall  f1-score   support

          -1      0.748     0.670     0.706       115
           0      0.783     0.728     0.755       114
           1      0.491     0.788     0.605        33

    accuracy                          0.710       262
   macro avg      0.674     0.729     0.689       262
weighted avg      0.731     0.710     0.715       262


📊 Accuracy Summary:
DTAI-KULeuven/robbert-v2-dutch-sentiment                          => 0.4198
pdelobelle/robbert-v2-dutch-base                                  => 0.3397
DTAI-KULeuven/robbertje-merged-dutch-sentiment                    => 0.3893
nlptown/bert-base-multilingual-uncased-sentiment                  => 0.6221
GroNLP/bert-base-dutch-cased                                      => 0.4198
citizenlab/twitter-xlm-roberta-base-sentiment-finetunned          => 0.6374
BramVanroy/xlm-roberta-base-hebban-reviews                        => 0.5305
BramVanroy/bert-base-multilingual-cased-hebban-

In [13]:
for model_name, acc in results.items():
    print(f"{model_name:65s} => {acc:.4f}")

DTAI-KULeuven/robbert-v2-dutch-sentiment                          => 0.4198
pdelobelle/robbert-v2-dutch-base                                  => 0.3397
DTAI-KULeuven/robbertje-merged-dutch-sentiment                    => 0.3893
nlptown/bert-base-multilingual-uncased-sentiment                  => 0.6221
GroNLP/bert-base-dutch-cased                                      => 0.4198
citizenlab/twitter-xlm-roberta-base-sentiment-finetunned          => 0.6374
BramVanroy/xlm-roberta-base-hebban-reviews                        => 0.5305
BramVanroy/bert-base-multilingual-cased-hebban-reviews            => 0.3740
BramVanroy/robbert-v2-dutch-base-hebban-reviews                   => 0.3893
clips/republic                                                    => 0.6985
tabularisai/multilingual-sentiment-analysis                       => 0.7099


In [15]:
results

{'DTAI-KULeuven/robbert-v2-dutch-sentiment': 0.4198473282442748,
 'pdelobelle/robbert-v2-dutch-base': 0.33969465648854963,
 'DTAI-KULeuven/robbertje-merged-dutch-sentiment': 0.3893129770992366,
 'nlptown/bert-base-multilingual-uncased-sentiment': 0.6221374045801527,
 'GroNLP/bert-base-dutch-cased': 0.4198473282442748,
 'citizenlab/twitter-xlm-roberta-base-sentiment-finetunned': 0.6374045801526718,
 'BramVanroy/xlm-roberta-base-hebban-reviews': 0.5305343511450382,
 'BramVanroy/bert-base-multilingual-cased-hebban-reviews': 0.37404580152671757,
 'BramVanroy/robbert-v2-dutch-base-hebban-reviews': 0.3893129770992366,
 'clips/republic': 0.6984732824427481,
 'tabularisai/multilingual-sentiment-analysis': 0.7099236641221374}

In [19]:
#### evaluation of top 3 perforingg models using majprty voting



import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")
from transformers import pipeline
import nltk
import pandas as pd
from collections import Counter
from tqdm import tqdm

nltk.download('punkt')

# Load the data
data =  carlijn_data_sent
data.replace('\n', pd.NA, inplace=True)

# Drop rows containing NaN values
#data.dropna(inplace=True)

# Reset index if needed
data.reset_index(drop=True, inplace=True)
print(len(data["Comments"]))

class zero_shot_voting():
    def __init__(self):
        # Initializing models for zero-shot sentiment classification
        self.models = [
            #'DTAI-KULeuven/robbert-v2-dutch-sentiment',  # Label Negative Positive
            #'DTAI-KULeuven/robbertje-merged-dutch-sentiment', # Negative or Positive
            'nlptown/bert-base-multilingual-uncased-sentiment', # Other models can be added
            "clips/republic",
            #"tabularisai/multilingual-sentiment-analysis"
            "citizenlab/twitter-xlm-roberta-base-sentiment-finetunned"
        ]
        self.classifiers = [pipeline(task="sentiment-analysis", model=model, return_all_scores=True, truncation=True, max_length=512) for model in self.models]

    def sentiment(self, prediction):
        # Converting all predictions into numerical format for consistency
        if(prediction=="Very Positive" or prediction=="Positive" or prediction == "LABEL_1" or prediction== '1' or prediction=="pos" or prediction=="4 stars" or prediction=="5 stars" or prediction=="positive"):
            return int(1)
        elif(prediction=="Very Negative" or prediction=="Negative" or prediction == "LABEL_0" or prediction=='-1' or prediction=="neg" or prediction=="1 star" or prediction=="negative" or prediction=="2 stars"):
            return int(-1)
        else:
            return int(0)

    def prediction(self, text):
        # Prediction from all models
        predictions = []
        prediction_scores = []
        for classifier in self.classifiers:
            result = classifier(text)[0]
            

            if(len(result) == 2):
                # Extract scores for positive and negative labels
                positive_score = result[1]['score']
                negative_score = result[0]['score']

                # Determine sentiment label based on scores
                if abs(positive_score - negative_score) <= 0.1:
                    sentiment_label = 'neutral'
                elif positive_score > negative_score:
                    sentiment_label = 'Positive'
                else:
                    sentiment_label = 'Negative'


                predictions.append(self.sentiment(sentiment_label))

            else:
                predictions.append(self.sentiment(max(result, key=lambda x: x['score'])['label']))
                prediction_scores.append(max(result, key=lambda x: x['score'])['score'])
        return predictions, prediction_scores

    def are_all_same(self, pred):
        return all(x == pred[0] for x in pred)

    def overall_sentiment_sentence(self, predictions):
        """
        Returns overall sentiment based on strict majority voting.
        If a label appears 2 or more times among model predictions, return that label.
        Else, return 0 (Neutral).
        """
        label_counts = Counter(predictions)

        for label, count in label_counts.items():
            if count >= 2:
                return label  # Strong majority found

        return 0  # Neutral if no label appears 2 or more times


    def sentiment_over_comment_majority(self, sentiment):
        if not sentiment:
            return 0  # Default to neutral if no sentiments

        counts = Counter(sentiment)
        most_common = max(counts.values(), default=0)  # Use default to avoid empty sequence error
        max_labels = [label for label, count in counts.items() if count == most_common]

        if len(max_labels) == 1:
            return max_labels[0]

        elif len(max_labels) == 2 and counts[max_labels[0]] == counts[max_labels[1]]:
            return 0  # Mixed sentiment case

        return 2  # Neutral for other ambiguous cases

    def sentiment_across_sentences(self, text):
        sentences = nltk.sent_tokenize(text, language='dutch')
        majority_voting_sentences = []
        overall_sentiment = []
        overall_sentiment_score = []
        for sentence in sentences:
            overall_sentiment1, overall_sentiment_score1 = self.prediction(sentence)
            overall_sentiment.append(overall_sentiment1)
            overall_sentiment_score.append(overall_sentiment_score1)
            majority_score = self.overall_sentiment_sentence(overall_sentiment1)
            majority_voting_sentences.append(majority_score)
        sentiment_over_comment = self.sentiment_over_comment_majority(majority_voting_sentences)
        return overall_sentiment, overall_sentiment_score, majority_voting_sentences, sentiment_over_comment


# DataFrame to store results
df = pd.DataFrame(columns=["Comment", "Sentiment of the four models", "Score", "Overall voted sentiment", "Overall Sentiment over comment from sentences", 
                           "Sentiment from models", "Sentiment from models score", "Sentiment over comment from models", "Score for entire comment"])

obj = zero_shot_voting()

# Process each comment
for i in tqdm(range(len(data["Comments"]))):
    overall_sentiment, overall_sentiment_score, overall_sentiment_voting, sentiment_over_comment = obj.sentiment_across_sentences(data["Comments"][i])

    # Sentiment over sending entire comment to models
    pred, pred_score = obj.prediction(data["Comments"][i])
    check = {
        
        'Comment': data["Comments"][i],
        'Sentiment of the four models': overall_sentiment,
        'Score': overall_sentiment_score,
        'Overall voted sentiment': overall_sentiment_voting,
        'Overall Sentiment over comment from sentences': sentiment_over_comment,
        "Sentiment from models": pred,
        "Sentiment from models score": pred_score,
        "Sentiment over comment from models": obj.overall_sentiment_sentence(pred),
        "Score for entire comment": pred_score
    }
    check = pd.DataFrame([check])
    df = pd.concat([df, check], ignore_index=True)





[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\20245179\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


262


Device set to use cpu
Device set to use cpu
Device set to use cpu
100%|██████████| 262/262 [01:39<00:00,  2.64it/s]


In [20]:
df

,Comment,Sentiment of the four models,Score,Overall voted sentiment,Overall Sentiment over comment from sentences,Sentiment from models,Sentiment from models score,Sentiment over comment from models,Score for entire comment
0,Je huur moet wel boven de 575 zijn ze verlagen...,"[[-1, 0, 0]]","[[0.3734073340892792, 0.6968942284584045, 0.98...",[0],0,"[-1, 0, 0]","[0.3734073340892792, 0.6968942284584045, 0.983...",0,"[0.3734073340892792, 0.6968942284584045, 0.983..."
1,Nemen jullie de telefoon nog op?,"[[-1, 0, 0]]","[[0.29925885796546936, 0.8234704732894897, 0.9...",[0],0,"[-1, 0, 0]","[0.29925885796546936, 0.8234704732894897, 0.97...",0,"[0.29925885796546936, 0.8234704732894897, 0.97..."
2,"Hopelijk zo'n project weer in Stratum, dan kun...","[[1, 0, 0]]","[[0.2372172474861145, 0.7590154409408569, 0.85...",[0],0,"[1, 0, 0]","[0.2372172474861145, 0.7590154409408569, 0.812...",0,"[0.2372172474861145, 0.7590154409408569, 0.812..."
3,Echt slecht je loopt alleen maar tegen problem...,"[[-1, -1, -1]]","[[0.9613326787948608, 0.9995256662368774, 0.96...",[-1],-1,"[-1, -1, -1]","[0.9613326787948608, 0.9995256662368774, 0.963...",-1,"[0.9613326787948608, 0.9995256662368774, 0.963..."
4,Zo zijn ze ook bij mijn moeder geweest... alth...,"[[0, -1, 0], [1, -1, 0], [1, 1, 1], [0, -1, 0]...","[[0.29710686206817627, 0.993301510810852, 0.99...","[0, 0, 1, 0, -1, 1]",0,"[1, 1, 0]","[0.5569653511047363, 0.92743980884552, 0.99056...",1,"[0.5569653511047363, 0.92743980884552, 0.99056..."
...,...,...,...,...,...,...,...,...,...
257,Dan kom ik langs voor sleutels en krijg er gra...,"[[0, 0, 0]]","[[0.29231977462768555, 0.982359766960144, 0.98...",[0],0,"[0, 0, 0]","[0.29231977462768555, 0.982359766960144, 0.989...",0,"[0.29231977462768555, 0.982359766960144, 0.989..."
258,Tussendeuren in huis gesloten houden ??? moet ...,"[[-1, -1, 0], [-1, -1, 0], [-1, -1, 0], [-1, -...","[[0.28364184498786926, 0.8375154137611389, 0.9...","[-1, -1, -1, -1, -1]",-1,"[-1, -1, 0]","[0.45288223028182983, 0.9982820749282837, 0.96...",-1,"[0.45288223028182983, 0.9982820749282837, 0.96..."
259,"Even een tip,volgens Dammers kloppen jullie pl...","[[-1, -1, 0]]","[[0.4621994197368622, 0.9996151924133301, 0.96...",[-1],-1,"[-1, -1, 0]","[0.4621994197368622, 0.9996151924133301, 0.967...",-1,"[0.4621994197368622, 0.9996151924133301, 0.967..."
260,Woonenergie regelt toch de sw woningbouwvereni...,"[[0, 0, 0]]","[[0.30479252338409424, 0.9894774556159973, 0.9...",[0],0,"[0, 0, 0]","[0.30479252338409424, 0.9894774556159973, 0.98...",0,"[0.30479252338409424, 0.9894774556159973, 0.98..."


In [21]:
to_check = pd.DataFrame()
to_check["Comment"] = data["Comments"]
to_check["pred_sentiment"] = df["Sentiment over comment from models"]
to_check["real_sentiment"] = data["PI_Sentiment"]

In [22]:
to_check

,Comment,pred_sentiment,real_sentiment
0,Je huur moet wel boven de 575 zijn ze verlagen...,0,0
1,Nemen jullie de telefoon nog op?,0,0
2,"Hopelijk zo'n project weer in Stratum, dan kun...",0,1
3,Echt slecht je loopt alleen maar tegen problem...,-1,-1
4,Zo zijn ze ook bij mijn moeder geweest... alth...,1,-1
...,...,...,...
257,Dan kom ik langs voor sleutels en krijg er gra...,0,0
258,Tussendeuren in huis gesloten houden ??? moet ...,-1,-1
259,"Even een tip,volgens Dammers kloppen jullie pl...",-1,0
260,Woonenergie regelt toch de sw woningbouwvereni...,0,0


In [23]:
y_true = [str(label) for label in to_check["real_sentiment"]]
y_pred = [str(label) for label in to_check["pred_sentiment"]]
import numpy as np
print(classification_report(y_true, y_pred, digits=3))

              precision    recall  f1-score   support

          -1      0.746     0.739     0.742       115
           0      0.758     0.658     0.704       114
           1      0.571     0.848     0.683        33

    accuracy                          0.718       262
   macro avg      0.692     0.749     0.710       262
weighted avg      0.729     0.718     0.718       262

